---
# <div style="text-align: center">Compare SCOPE and RDKit molecular overlaps</div>
---


This notebook imports the selected cell2mol `Cell` objects and compares equivalent molecules using SCOPE and two RDKit implementations: one constructed from atom labels and coordinates only, and one that also uses the SCOPE bond data. It records the elapsed time, RMSD, maximum corresponding-atom displacement, and `mol.iscomplex` classification obtained for each comparison in `molecule_comparison_results.csv`.

SCOPE is compared with each RDKit implementation separately. Cases where the symmetric percentage difference between SCOPE and either RDKit RMSD reaches `rmsd_percentage_threshold` are retained with their original and aligned coordinates for manual inspection. Import validation is performed separately in Notebook 1.

The SCOPE comparison uses the connectivity and bond orders already stored in each imported `Molecule`; preparing that topology is excluded from the elapsed overlap time, like construction of the RDKit molecules.


## Part 0. Configure the dataset and load the selected paths

The cell2mol dataset is stored outside the repository. Set `datasets_folder` to its parent folder, using the same folder structure described in Notebook 1. The provided `selected_cell_paths.txt` fixes the 25 Cells used to obtain the 201 successful molecular comparisons stored in `molecule_comparison_results.csv`; no new random selection is performed.


In [1]:
import csv
import os
import sys
import time

import numpy as np
import scope

from rdkit import Chem, rdBase
from rdkit.Chem import rdMolAlign
from scope.classes_cell import import_cell

sys.path.insert(0, os.path.abspath('..'))
from benchmark_functions import rdkit_overlap, scope_overlap


In [2]:
# Datasets are not uploaded to the repository. Set their parent folder here.
benchmark_folder        = os.path.abspath('./')
datasets_folder         = os.path.abspath('/Volumes/Science/Datasets')
cell2mol_dataset_folder = os.path.join(datasets_folder, 'cell2mol')
cell_selection_file     = os.path.join(benchmark_folder, 'selected_cell_paths.txt')

if not os.path.isdir(cell2mol_dataset_folder): raise FileNotFoundError(f'Cell dataset not found at {cell2mol_dataset_folder}')
if not os.path.isfile(cell_selection_file): raise FileNotFoundError(f'Cell selection not found at {cell_selection_file}')

with open(cell_selection_file, 'r', encoding='utf-8') as selected_file:
    relative_cell_paths = [line.strip() for line in selected_file if line.strip()]
cell_paths = [os.path.join(cell2mol_dataset_folder, path) for path in relative_cell_paths]
assert len(set(cell_paths)) == len(cell_paths), 'The selected path list contains repeated entries'
assert all(os.path.isfile(path) for path in cell_paths), 'At least one selected Cell file is missing'
print('Selected Cell files:', len(relative_cell_paths))

Selected Cell files: 25


## Part 1. Overlap molecules with RDKit


In [3]:
from rdkit import Chem
from rdkit.Chem import rdMolAlign

def build_rdkit_mol(atom_labels, coords):
    """
    atom_labels: list of strings, e.g. ['C', 'H', 'H', 'O']
    coords: np.array of shape (N_atoms, 3)
    """
    mol = Chem.RWMol()  # editable molecule
    # add atoms
    for label in atom_labels:
        atom = Chem.Atom(label)
        mol.AddAtom(atom)
    # create a conformer (3D structure)
    conf = Chem.Conformer(len(atom_labels))
    for i, (x, y, z) in enumerate(coords):
        conf.SetAtomPosition(i, Chem.rdGeometry.Point3D(x, y, z))
    mol.AddConformer(conf)
    return mol

def build_bonded_rdkit_mol(molecule):
    rdkit_mol = Chem.RWMol()
    for label in molecule.labels:
        rdkit_mol.AddAtom(Chem.Atom(label))

    if not hasattr(molecule, 'atoms'):
        molecule.set_atoms()
    if not hasattr(molecule, 'adjmat'):
        molecule.get_adjmatrix()
    if not hasattr(molecule, 'madjmat'):
        molecule.get_metal_adjmatrix()

    bond_orders = {}
    for atom in molecule.atoms:
        for bond in atom.bonds:
            atom_index1 = bond.atom1.get_parent_index('molecule')
            atom_index2 = bond.atom2.get_parent_index('molecule')
            bond_key = tuple(sorted((atom_index1, atom_index2)))
            bond_orders[bond_key] = bond.order

    metal_adjacency = molecule.madjmat
    if metal_adjacency is None:
        metal_adjacency = np.zeros_like(molecule.adjmat)

    for atom_index1 in range(molecule.natoms):
        for atom_index2 in range(atom_index1 + 1, molecule.natoms):
            is_covalent_bond = molecule.adjmat[atom_index1, atom_index2] > 0
            is_metal_bond = metal_adjacency[atom_index1, atom_index2] > 0
            if not is_covalent_bond and not is_metal_bond:
                continue

            bond_begin = atom_index1
            bond_end = atom_index2
            bond_order = bond_orders.get((atom_index1, atom_index2), 1.0)

            if is_metal_bond or np.isclose(bond_order, 0.5):
                if molecule.atoms[atom_index1].object_subtype == 'metal':
                    bond_begin, bond_end = atom_index2, atom_index1
                rdkit_bond_type = Chem.BondType.DATIVE
            elif np.isclose(bond_order, 1.0):
                rdkit_bond_type = Chem.BondType.SINGLE
            elif np.isclose(bond_order, 1.5):
                rdkit_bond_type = Chem.BondType.AROMATIC
            elif np.isclose(bond_order, 2.0):
                rdkit_bond_type = Chem.BondType.DOUBLE
            elif np.isclose(bond_order, 3.0):
                rdkit_bond_type = Chem.BondType.TRIPLE
            else:
                raise ValueError(f'Unsupported SCOPE bond order: {bond_order}')

            rdkit_mol.AddBond(bond_begin, bond_end, rdkit_bond_type)
            if rdkit_bond_type == Chem.BondType.AROMATIC:
                rdkit_mol.GetAtomWithIdx(bond_begin).SetIsAromatic(True)
                rdkit_mol.GetAtomWithIdx(bond_end).SetIsAromatic(True)

    conformer = Chem.Conformer(molecule.natoms)
    for atom_index, (x, y, z) in enumerate(molecule.coord):
        conformer.SetAtomPosition(atom_index, Chem.rdGeometry.Point3D(x, y, z))
    rdkit_mol.AddConformer(conformer)
    return rdkit_mol


## Part 2. Import Cells and compare equivalent molecules

The `rmsd_percentage_threshold` is which difference do we consider to be significant enough to establish that one method is faster than the other. It is calculated as `100 × |SCOPE − RDKit| / mean(SCOPE, RDKit)`. If a difference is below that threshold, we assume the same timing.


In [4]:
rmsd_percentage_threshold = 25.0
comparison_file_path = os.path.join(benchmark_folder, 'molecule_comparison_results.csv')
comparison_file = open(comparison_file_path, 'w', newline='', encoding='utf-8')
comparison_writer = csv.writer(comparison_file)
comparison_writer.writerow([
    'cell_path', 'molecule_index_1', 'molecule_index_2', 'formula', 'number_of_atoms', 'iscomplex',
    'scope_time_s', 'rdkit_bondless_time_s', 'rdkit_bonded_time_s',
    'scope_rmsd_angstrom', 'rdkit_bondless_rmsd_angstrom', 'rdkit_bonded_rmsd_angstrom',
    'scope_max_atom_distance_angstrom',
    'rdkit_bondless_max_atom_distance_angstrom', 'rdkit_bonded_max_atom_distance_angstrom',
    'scope_max_atom_index',
    'rdkit_bondless_max_mobile_atom_index', 'rdkit_bondless_max_reference_atom_index',
    'rdkit_bonded_max_mobile_atom_index', 'rdkit_bonded_max_reference_atom_index',
    'scope_rdkit_bondless_rmsd_difference_angstrom', 'scope_rdkit_bondless_rmsd_difference_percent',
    'scope_rdkit_bonded_rmsd_difference_angstrom', 'scope_rdkit_bonded_rmsd_difference_percent',
    'faster_method_scope_vs_rdkit_bondless', 'faster_method_scope_vs_rdkit_bonded',
    'lower_rmsd_method_scope_vs_rdkit_bondless', 'lower_rmsd_method_scope_vs_rdkit_bonded',
    'lower_max_distance_method_scope_vs_rdkit_bondless',
    'lower_max_distance_method_scope_vs_rdkit_bonded'
])
comparison_file.flush()

scope_times = []
rdkit_bondless_times = []
rdkit_bonded_times = []
scope_rmsd_values = []
rdkit_bondless_rmsd_values = []
rdkit_bonded_rmsd_values = []
scope_maximum_distances = []
rdkit_bondless_maximum_distances = []
rdkit_bonded_maximum_distances = []
iscomplex_values = []
large_rmsd_differences = []
failed_imports = []
failed_comparisons = []
imported_cells = 0
compared_molecules = 0

for cell_index, relative_path in enumerate(relative_cell_paths):
    cell_path = os.path.join(cell2mol_dataset_folder, relative_path)

    try:
        cell2mol_cell = scope.load_binary(cell_path)
        with rdBase.BlockLogs():
            imported_cell = import_cell(cell2mol_cell)
        if imported_cell is None:
            raise ValueError('SCOPE did not import this Cell because cell2mol reported warnings')
        imported_cells += 1
    except Exception as error:
        failed_imports.append((relative_path, str(error)))
        print(f'Failed import: {relative_path}: {error}')
        continue

    molecule_pairs = []
    for molecule_index1, molecule1 in enumerate(imported_cell.molecules):
        if molecule1.natoms < 10:
            continue
        for molecule_index2, molecule2 in enumerate(imported_cell.molecules):
            if molecule_index2 <= molecule_index1 or molecule1 != molecule2:
                continue
            molecule_pairs.append((molecule_index1, molecule1, molecule_index2, molecule2))

    rdkit_results = []
    rdkit_failed  = False
    for molecule_index1, molecule1, molecule_index2, molecule2 in molecule_pairs:
        try:
            rdkit_bondless_mol1 = build_rdkit_mol(molecule1.labels, molecule1.coord)
            rdkit_bondless_mol2 = build_rdkit_mol(molecule2.labels, molecule2.coord)
            rdkit_bonded_mol1   = build_bonded_rdkit_mol(molecule1)
            rdkit_bonded_mol2   = build_bonded_rdkit_mol(molecule2)
            rdkit_bondless_time, rdkit_bondless_rmsd, rdkit_bondless_maximum_distance, rdkit_bondless_maximum_distance_atom_pair, rdkit_bondless_coord = rdkit_overlap(rdkit_bondless_mol1, rdkit_bondless_mol2)
            rdkit_bonded_time, rdkit_bonded_rmsd, rdkit_bonded_maximum_distance, rdkit_bonded_maximum_distance_atom_pair, rdkit_bonded_coord = rdkit_overlap(rdkit_bonded_mol1, rdkit_bonded_mol2)
        except Exception as error:
            failed_comparisons.append((relative_path, molecule_index1, molecule_index2, f'RDKit: {error}'))
            print(f'RDKit failed for molecules {molecule_index1} and {molecule_index2}. Skipping Cell: {relative_path}: {error}')
            rdkit_failed = True
            break
        rdkit_results.append((molecule_index1, molecule1, molecule_index2, molecule2, rdkit_bondless_time, rdkit_bondless_rmsd, rdkit_bondless_maximum_distance, rdkit_bondless_maximum_distance_atom_pair, rdkit_bondless_coord, rdkit_bonded_time, rdkit_bonded_rmsd, rdkit_bonded_maximum_distance, rdkit_bonded_maximum_distance_atom_pair, rdkit_bonded_coord))

    if not rdkit_failed:
        for rdkit_result in rdkit_results:
            molecule_index1, molecule1, molecule_index2, molecule2, rdkit_bondless_time, rdkit_bondless_rmsd, rdkit_bondless_maximum_distance, rdkit_bondless_maximum_distance_atom_pair, rdkit_bondless_coord, rdkit_bonded_time, rdkit_bonded_rmsd, rdkit_bonded_maximum_distance, rdkit_bonded_maximum_distance_atom_pair, rdkit_bonded_coord = rdkit_result
            try:
                scope_time, scope_rmsd, scope_maximum_distance, scope_maximum_distance_index, scope_coord = scope_overlap(molecule1, molecule2)
            except Exception as error:
                failed_comparisons.append((relative_path, molecule_index1, molecule_index2, f'SCOPE: {error}'))
                continue

            compared_molecules += 1
            scope_times.append(scope_time)
            rdkit_bondless_times.append(rdkit_bondless_time)
            rdkit_bonded_times.append(rdkit_bonded_time)
            scope_rmsd_values.append(scope_rmsd)
            rdkit_bondless_rmsd_values.append(rdkit_bondless_rmsd)
            rdkit_bonded_rmsd_values.append(rdkit_bonded_rmsd)
            scope_maximum_distances.append(scope_maximum_distance)
            rdkit_bondless_maximum_distances.append(rdkit_bondless_maximum_distance)
            rdkit_bonded_maximum_distances.append(rdkit_bonded_maximum_distance)
            iscomplex_values.append(molecule1.iscomplex)

            faster_than_bondless = 'SCOPE' if scope_time < rdkit_bondless_time else 'RDKit without bonds'
            faster_than_bonded = 'SCOPE' if scope_time < rdkit_bonded_time else 'RDKit with bonds'

            if np.isclose(scope_rmsd, rdkit_bondless_rmsd, atol=1e-4):
                lower_rmsd_than_bondless = 'Equal'
            elif scope_rmsd < rdkit_bondless_rmsd:
                lower_rmsd_than_bondless = 'SCOPE'
            else:
                lower_rmsd_than_bondless = 'RDKit without bonds'

            if np.isclose(scope_rmsd, rdkit_bonded_rmsd, atol=1e-4):
                lower_rmsd_than_bonded = 'Equal'
            elif scope_rmsd < rdkit_bonded_rmsd:
                lower_rmsd_than_bonded = 'SCOPE'
            else:
                lower_rmsd_than_bonded = 'RDKit with bonds'

            if np.isclose(scope_maximum_distance, rdkit_bondless_maximum_distance, atol=1e-4):
                lower_maximum_distance_than_bondless = 'Equal'
            elif scope_maximum_distance < rdkit_bondless_maximum_distance:
                lower_maximum_distance_than_bondless = 'SCOPE'
            else:
                lower_maximum_distance_than_bondless = 'RDKit without bonds'

            if np.isclose(scope_maximum_distance, rdkit_bonded_maximum_distance, atol=1e-4):
                lower_maximum_distance_than_bonded = 'Equal'
            elif scope_maximum_distance < rdkit_bonded_maximum_distance:
                lower_maximum_distance_than_bonded = 'SCOPE'
            else:
                lower_maximum_distance_than_bonded = 'RDKit with bonds'

            if np.isclose(scope_rmsd, rdkit_bondless_rmsd, atol=1e-4):
                bondless_rmsd_percentage_difference = 0.0
            else:
                mean_rmsd = (scope_rmsd + rdkit_bondless_rmsd) / 2
                bondless_rmsd_percentage_difference = 100 * abs(scope_rmsd - rdkit_bondless_rmsd) / mean_rmsd

            if np.isclose(scope_rmsd, rdkit_bonded_rmsd, atol=1e-4):
                bonded_rmsd_percentage_difference = 0.0
            else:
                mean_rmsd = (scope_rmsd + rdkit_bonded_rmsd) / 2
                bonded_rmsd_percentage_difference = 100 * abs(scope_rmsd - rdkit_bonded_rmsd) / mean_rmsd

            comparison_writer.writerow([
                relative_path, molecule_index1, molecule_index2, molecule1.formula, molecule1.natoms, molecule1.iscomplex,
                scope_time, rdkit_bondless_time, rdkit_bonded_time,
                scope_rmsd, rdkit_bondless_rmsd, rdkit_bonded_rmsd,
                scope_maximum_distance, rdkit_bondless_maximum_distance, rdkit_bonded_maximum_distance,
                scope_maximum_distance_index,
                rdkit_bondless_maximum_distance_atom_pair[0], rdkit_bondless_maximum_distance_atom_pair[1],
                rdkit_bonded_maximum_distance_atom_pair[0], rdkit_bonded_maximum_distance_atom_pair[1],
                abs(scope_rmsd - rdkit_bondless_rmsd), bondless_rmsd_percentage_difference,
                abs(scope_rmsd - rdkit_bonded_rmsd), bonded_rmsd_percentage_difference,
                faster_than_bondless, faster_than_bonded,
                lower_rmsd_than_bondless, lower_rmsd_than_bonded,
                lower_maximum_distance_than_bondless, lower_maximum_distance_than_bonded
            ])
            comparison_file.flush()

            if max(bondless_rmsd_percentage_difference, bonded_rmsd_percentage_difference) >= rmsd_percentage_threshold:
                large_rmsd_differences.append({
                    'cell_path': relative_path,
                    'molecule_indices': (molecule_index1, molecule_index2),
                    'formula': molecule1.formula,
                    'iscomplex': molecule1.iscomplex,
                    'scope_rmsd': scope_rmsd,
                    'rdkit_bondless_rmsd': rdkit_bondless_rmsd,
                    'rdkit_bonded_rmsd': rdkit_bonded_rmsd,
                    'scope_rdkit_bondless_rmsd_percentage_difference': bondless_rmsd_percentage_difference,
                    'scope_rdkit_bonded_rmsd_percentage_difference': bonded_rmsd_percentage_difference,
                    'scope_maximum_atom_distance': scope_maximum_distance,
                    'rdkit_bondless_maximum_atom_distance': rdkit_bondless_maximum_distance,
                    'rdkit_bonded_maximum_atom_distance': rdkit_bonded_maximum_distance,
                    'scope_maximum_atom_index': scope_maximum_distance_index,
                    'rdkit_bondless_maximum_atom_pair': rdkit_bondless_maximum_distance_atom_pair,
                    'rdkit_bonded_maximum_atom_pair': rdkit_bonded_maximum_distance_atom_pair,
                    'scope_time': scope_time,
                    'rdkit_bondless_time': rdkit_bondless_time,
                    'rdkit_bonded_time': rdkit_bonded_time,
                    'reference_labels': molecule1.labels.copy(),
                    'reference_coordinates': np.asarray(molecule1.coord).copy(),
                    'original_labels': molecule2.labels.copy(),
                    'original_coordinates': np.asarray(molecule2.coord).copy(),
                    'scope_labels': molecule1.labels.copy(),
                    'scope_coordinates': scope_coord.copy(),
                    'rdkit_bondless_labels': molecule2.labels.copy(),
                    'rdkit_bondless_coordinates': rdkit_bondless_coord.copy(),
                    'rdkit_bonded_labels': molecule2.labels.copy(),
                    'rdkit_bonded_coordinates': rdkit_bonded_coord.copy(),
                })

    if (cell_index + 1) % 25 == 0 or cell_index + 1 == len(relative_cell_paths):
        print(f'Processed {cell_index + 1}/{len(relative_cell_paths)} Cells')

comparison_file.close()


RDKit failed for molecules 2 and 6. Skipping Cell: 8-Copper/Cell_MODSIJ.gmol: No sub-structure match found between the reference and probe mol
RDKit failed for molecules 4 and 5. Skipping Cell: 7-Nickel/Cell_NIMNIH.gmol: No sub-structure match found between the reference and probe mol
RDKit failed for molecules 0 and 2. Skipping Cell: 1-Iron/Cell_ROVLOG.gmol: No sub-structure match found between the reference and probe mol
Processed 25/25 Cells


## Part 3. Comparison summary


In [5]:
print('Requested Cells:', len(relative_cell_paths))
print('Imported Cells:', imported_cells)
print('Failed imports:', len(failed_imports))
print('Compared molecule pairs:', compared_molecules)
print('Failed comparisons:', len(failed_comparisons))
print('Large RMSD differences:', len(large_rmsd_differences))
print('Comparison CSV:', comparison_file_path)

if compared_molecules:
    scope_times = np.asarray(scope_times)
    rdkit_bondless_times = np.asarray(rdkit_bondless_times)
    rdkit_bonded_times = np.asarray(rdkit_bonded_times)
    scope_rmsd_values = np.asarray(scope_rmsd_values)
    rdkit_bondless_rmsd_values = np.asarray(rdkit_bondless_rmsd_values)
    rdkit_bonded_rmsd_values = np.asarray(rdkit_bonded_rmsd_values)
    scope_maximum_distances = np.asarray(scope_maximum_distances)
    rdkit_bondless_maximum_distances = np.asarray(rdkit_bondless_maximum_distances)
    rdkit_bonded_maximum_distances = np.asarray(rdkit_bonded_maximum_distances)
    iscomplex_values = np.asarray(iscomplex_values, dtype=bool)

    rdkit_methods = [
        ('RDKit without bonds', rdkit_bondless_times, rdkit_bondless_rmsd_values, rdkit_bondless_maximum_distances),
        ('RDKit with bonds', rdkit_bonded_times, rdkit_bonded_rmsd_values, rdkit_bonded_maximum_distances),
    ]

    for rdkit_method, rdkit_times, rdkit_rmsd_values, rdkit_maximum_distances in rdkit_methods:
        scope_faster = np.sum(scope_times < rdkit_times)
        rdkit_faster = np.sum(rdkit_times < scope_times)
        equal_times = compared_molecules - scope_faster - rdkit_faster

        equal_rmsd = np.isclose(scope_rmsd_values, rdkit_rmsd_values, atol=1e-4)
        scope_lower_rmsd = np.sum((scope_rmsd_values < rdkit_rmsd_values) & ~equal_rmsd)
        rdkit_lower_rmsd = np.sum((rdkit_rmsd_values < scope_rmsd_values) & ~equal_rmsd)

        equal_maximum_distance = np.isclose(scope_maximum_distances, rdkit_maximum_distances, atol=1e-4)
        scope_lower_maximum_distance = np.sum((scope_maximum_distances < rdkit_maximum_distances) & ~equal_maximum_distance)
        rdkit_lower_maximum_distance = np.sum((rdkit_maximum_distances < scope_maximum_distances) & ~equal_maximum_distance)

        print(f'\nSCOPE vs {rdkit_method}')
        print(f'SCOPE mean time: {np.mean(scope_times):.6f} s')
        print(f'{rdkit_method} mean time: {np.mean(rdkit_times):.6f} s')
        print(f'SCOPE median time: {np.median(scope_times):.6f} s')
        print(f'{rdkit_method} median time: {np.median(rdkit_times):.6f} s')
        print(f'SCOPE faster: {scope_faster}/{compared_molecules} ({100 * scope_faster / compared_molecules:.1f}%)')
        print(f'{rdkit_method} faster: {rdkit_faster}/{compared_molecules} ({100 * rdkit_faster / compared_molecules:.1f}%)')
        print(f'Equal times: {equal_times}/{compared_molecules}')
        print(f'SCOPE mean RMSD: {np.mean(scope_rmsd_values):.6f} Angstrom')
        print(f'{rdkit_method} mean RMSD: {np.mean(rdkit_rmsd_values):.6f} Angstrom')
        print(f'SCOPE lower RMSD: {scope_lower_rmsd}/{compared_molecules} ({100 * scope_lower_rmsd / compared_molecules:.1f}%)')
        print(f'{rdkit_method} lower RMSD: {rdkit_lower_rmsd}/{compared_molecules} ({100 * rdkit_lower_rmsd / compared_molecules:.1f}%)')
        print(f'Equivalent RMSD: {np.sum(equal_rmsd)}/{compared_molecules}')
        print(f'SCOPE mean maximum atom distance: {np.mean(scope_maximum_distances):.6f} Angstrom')
        print(f'{rdkit_method} mean maximum atom distance: {np.mean(rdkit_maximum_distances):.6f} Angstrom')
        print(f'SCOPE lower maximum atom distance: {scope_lower_maximum_distance}/{compared_molecules} ({100 * scope_lower_maximum_distance / compared_molecules:.1f}%)')
        print(f'{rdkit_method} lower maximum atom distance: {rdkit_lower_maximum_distance}/{compared_molecules} ({100 * rdkit_lower_maximum_distance / compared_molecules:.1f}%)')
        print(f'Equivalent maximum atom distance: {np.sum(equal_maximum_distance)}/{compared_molecules}')

        for iscomplex, molecule_type in [(True, 'Metal complexes'), (False, 'Pure organic')]:
            molecule_type_mask = iscomplex_values == iscomplex
            molecule_type_comparisons = int(np.sum(molecule_type_mask))
            if molecule_type_comparisons == 0:
                print(f'\n{molecule_type}: no comparisons')
                continue

            type_scope_faster = np.sum(scope_times[molecule_type_mask] < rdkit_times[molecule_type_mask])
            type_rdkit_faster = np.sum(rdkit_times[molecule_type_mask] < scope_times[molecule_type_mask])
            type_equal_times = molecule_type_comparisons - type_scope_faster - type_rdkit_faster

            type_equal_rmsd = equal_rmsd[molecule_type_mask]
            type_scope_lower_rmsd = np.sum(
                (scope_rmsd_values[molecule_type_mask] < rdkit_rmsd_values[molecule_type_mask]) & ~type_equal_rmsd
            )
            type_rdkit_lower_rmsd = np.sum(
                (rdkit_rmsd_values[molecule_type_mask] < scope_rmsd_values[molecule_type_mask]) & ~type_equal_rmsd
            )

            type_equal_maximum_distance = equal_maximum_distance[molecule_type_mask]
            type_scope_lower_maximum_distance = np.sum(
                (scope_maximum_distances[molecule_type_mask] < rdkit_maximum_distances[molecule_type_mask])
                & ~type_equal_maximum_distance
            )
            type_rdkit_lower_maximum_distance = np.sum(
                (rdkit_maximum_distances[molecule_type_mask] < scope_maximum_distances[molecule_type_mask])
                & ~type_equal_maximum_distance
            )

            print(f'\n{molecule_type}: {molecule_type_comparisons} comparisons')
            print(f'  SCOPE mean time: {np.mean(scope_times[molecule_type_mask]):.6f} s')
            print(f'  {rdkit_method} mean time: {np.mean(rdkit_times[molecule_type_mask]):.6f} s')
            print(f'  SCOPE faster: {type_scope_faster}/{molecule_type_comparisons} ({100 * type_scope_faster / molecule_type_comparisons:.1f}%)')
            print(f'  {rdkit_method} faster: {type_rdkit_faster}/{molecule_type_comparisons} ({100 * type_rdkit_faster / molecule_type_comparisons:.1f}%)')
            print(f'  Equal times: {type_equal_times}/{molecule_type_comparisons}')
            print(f'  SCOPE mean RMSD: {np.mean(scope_rmsd_values[molecule_type_mask]):.6f} Angstrom')
            print(f'  {rdkit_method} mean RMSD: {np.mean(rdkit_rmsd_values[molecule_type_mask]):.6f} Angstrom')
            print(f'  SCOPE lower RMSD: {type_scope_lower_rmsd}/{molecule_type_comparisons} ({100 * type_scope_lower_rmsd / molecule_type_comparisons:.1f}%)')
            print(f'  {rdkit_method} lower RMSD: {type_rdkit_lower_rmsd}/{molecule_type_comparisons} ({100 * type_rdkit_lower_rmsd / molecule_type_comparisons:.1f}%)')
            print(f'  Equivalent RMSD: {np.sum(type_equal_rmsd)}/{molecule_type_comparisons}')
            print(f'  SCOPE mean maximum atom distance: {np.mean(scope_maximum_distances[molecule_type_mask]):.6f} Angstrom')
            print(f'  {rdkit_method} mean maximum atom distance: {np.mean(rdkit_maximum_distances[molecule_type_mask]):.6f} Angstrom')
            print(f'  SCOPE lower maximum atom distance: {type_scope_lower_maximum_distance}/{molecule_type_comparisons} ({100 * type_scope_lower_maximum_distance / molecule_type_comparisons:.1f}%)')
            print(f'  {rdkit_method} lower maximum atom distance: {type_rdkit_lower_maximum_distance}/{molecule_type_comparisons} ({100 * type_rdkit_lower_maximum_distance / molecule_type_comparisons:.1f}%)')
            print(f'  Equivalent maximum atom distance: {np.sum(type_equal_maximum_distance)}/{molecule_type_comparisons}')


Requested Cells: 25
Imported Cells: 25
Failed imports: 0
Compared molecule pairs: 201
Failed comparisons: 3
Large RMSD differences: 167
Comparison CSV: /Users/sergivela/Documents/SCOPE/Program/Scope/benchmarks/3-Molecule_Overlap/molecule_comparison_results.csv

SCOPE vs RDKit without bonds
SCOPE mean time: 0.067871 s
RDKit without bonds mean time: 1.820077 s
SCOPE median time: 0.020259 s
RDKit without bonds median time: 1.862437 s
SCOPE faster: 201/201 (100.0%)
RDKit without bonds faster: 0/201 (0.0%)
Equal times: 0/201
SCOPE mean RMSD: 0.604672 Angstrom
RDKit without bonds mean RMSD: 2.536209 Angstrom
SCOPE lower RMSD: 175/201 (87.1%)
RDKit without bonds lower RMSD: 4/201 (2.0%)
Equivalent RMSD: 22/201
SCOPE mean maximum atom distance: 1.467627 Angstrom
RDKit without bonds mean maximum atom distance: 5.545993 Angstrom
SCOPE lower maximum atom distance: 174/201 (86.6%)
RDKit without bonds lower maximum atom distance: 5/201 (2.5%)
Equivalent maximum atom distance: 22/201

Metal complexe

`molecule_comparison_results.csv` contains one row per successful comparison. SCOPE is compared separately with RDKit without bonds and RDKit with bonds, with independent elapsed times, RMSDs, maximum corresponding-atom displacements, atom indices, percentage differences, and winner columns. The file is flushed after every row and records whether `mol.iscomplex` classifies each molecule as a metal complex.

`large_rmsd_differences` retains the reference, original, and all three aligned geometries whenever either SCOPE–RDKit comparison reaches the RMSD percentage threshold. `failed_imports` and `failed_comparisons` retain cases that require separate inspection.


## Debug

Review a specific comparison from the CSV file, selected with its row index. It creates a new folder called "overlap_comparison_tests"

In [6]:
comparison_index = 13
comparison_file_path = os.path.join(benchmark_folder, 'molecule_comparison_results.csv')

with open(comparison_file_path, 'r', encoding='utf-8') as comparison_results_file:
    comparison_rows = list(csv.DictReader(comparison_results_file))

comparison = comparison_rows[comparison_index]
relative_path = comparison['cell_path']
molecule_index1 = int(comparison['molecule_index_1'])
molecule_index2 = int(comparison['molecule_index_2'])

cell_path = os.path.join(cell2mol_dataset_folder, relative_path)
cell2mol_cell = scope.load_binary(cell_path)
with rdBase.BlockLogs():
    imported_cell = import_cell(cell2mol_cell)

reference_molecule = imported_cell.molecules[molecule_index1]
original_molecule = imported_cell.molecules[molecule_index2]

_, scope_rmsd, scope_maximum_distance, scope_maximum_distance_index, scope_coord = scope_overlap(reference_molecule, original_molecule)
rdkit_bondless_mol1 = build_rdkit_mol(reference_molecule.labels, reference_molecule.coord)
rdkit_bondless_mol2 = build_rdkit_mol(original_molecule.labels, original_molecule.coord)
rdkit_bonded_mol1   = build_bonded_rdkit_mol(reference_molecule)
rdkit_bonded_mol2   = build_bonded_rdkit_mol(original_molecule)
_, rdkit_bondless_rmsd, rdkit_bondless_maximum_distance, rdkit_bondless_maximum_distance_atom_pair, rdkit_bondless_coord = rdkit_overlap(rdkit_bondless_mol1, rdkit_bondless_mol2)
_, rdkit_bonded_rmsd, rdkit_bonded_maximum_distance, rdkit_bonded_maximum_distance_atom_pair, rdkit_bonded_coord = rdkit_overlap(rdkit_bonded_mol1, rdkit_bonded_mol2)

from scope.classes_specie import Molecule

scope_overlapped_molecule = Molecule(reference_molecule.labels.copy(), scope_coord.tolist())
rdkit_bondless_overlapped_molecule = Molecule(original_molecule.labels.copy(), rdkit_bondless_coord.tolist())
rdkit_bonded_overlapped_molecule = Molecule(original_molecule.labels.copy(), rdkit_bonded_coord.tolist())

cell_name = os.path.splitext(os.path.basename(relative_path))[0]
comparison_name = f'{cell_name}_molecules_{molecule_index1}_{molecule_index2}'
comparison_folder = os.path.join(benchmark_folder, 'overlap_comparison_tests', comparison_name)
os.makedirs(comparison_folder, exist_ok=True)

scope.write_xyz(
    os.path.join(comparison_folder, 'reference_molecule.xyz'),
    reference_molecule.labels, reference_molecule.coord,
    charge=reference_molecule.charge, spin=reference_molecule.spin_multiplicity
)
scope.write_xyz(
    os.path.join(comparison_folder, 'original_molecule.xyz'),
    original_molecule.labels, original_molecule.coord,
    charge=original_molecule.charge, spin=original_molecule.spin_multiplicity
)
scope.write_xyz(
    os.path.join(comparison_folder, 'scope_overlapped_molecule.xyz'),
    scope_overlapped_molecule.labels, scope_overlapped_molecule.coord,
    charge=original_molecule.charge, spin=original_molecule.spin_multiplicity
)
scope.write_xyz(
    os.path.join(comparison_folder, 'rdkit_bondless_overlapped_molecule.xyz'),
    rdkit_bondless_overlapped_molecule.labels, rdkit_bondless_overlapped_molecule.coord,
    charge=original_molecule.charge, spin=original_molecule.spin_multiplicity
)
scope.write_xyz(
    os.path.join(comparison_folder, 'rdkit_bonded_overlapped_molecule.xyz'),
    rdkit_bonded_overlapped_molecule.labels, rdkit_bonded_overlapped_molecule.coord,
    charge=original_molecule.charge, spin=original_molecule.spin_multiplicity
)

print('Cell:', relative_path)
print('Molecule indices:', molecule_index1, molecule_index2)
print('Formula:', comparison['formula'])
print('Molecule type:', 'Metal complex' if reference_molecule.iscomplex else 'Pure organic')
print('CSV iscomplex:', comparison['iscomplex'])
print('CSV SCOPE RMSD:', comparison['scope_rmsd_angstrom'])
print('Recomputed SCOPE RMSD:', scope_rmsd)
print('CSV RDKit without bonds RMSD:', comparison['rdkit_bondless_rmsd_angstrom'])
print('Recomputed RDKit without bonds RMSD:', rdkit_bondless_rmsd)
print('CSV RDKit with bonds RMSD:', comparison['rdkit_bonded_rmsd_angstrom'])
print('Recomputed RDKit with bonds RMSD:', rdkit_bonded_rmsd)
print('CSV SCOPE maximum atom distance:', comparison['scope_max_atom_distance_angstrom'])
print('Recomputed SCOPE maximum atom distance:', scope_maximum_distance, 'at atom', scope_maximum_distance_index)
print('CSV RDKit without bonds maximum atom distance:', comparison['rdkit_bondless_max_atom_distance_angstrom'])
print('Recomputed RDKit without bonds maximum atom distance:', rdkit_bondless_maximum_distance, 'for mobile/reference atom pair', rdkit_bondless_maximum_distance_atom_pair)
print('CSV RDKit with bonds maximum atom distance:', comparison['rdkit_bonded_max_atom_distance_angstrom'])
print('Recomputed RDKit with bonds maximum atom distance:', rdkit_bonded_maximum_distance, 'for mobile/reference atom pair', rdkit_bonded_maximum_distance_atom_pair)
print('XYZ folder:', comparison_folder)
print('Available structures: reference_molecule, original_molecule, scope_overlapped_molecule, rdkit_bondless_overlapped_molecule, rdkit_bonded_overlapped_molecule')


Cell: 7-Nickel/Cell_POMHIJ.gmol
Molecule indices: 1 7
Formula: H6-C7-O2
Molecule type: Pure organic
CSV iscomplex: False
CSV SCOPE RMSD: 0.09769612853679678
Recomputed SCOPE RMSD: 0.09769612853679678
CSV RDKit without bonds RMSD: 0.09769612853597351
Recomputed RDKit without bonds RMSD: 0.09769612853597351
CSV RDKit with bonds RMSD: 0.09769612853597351
Recomputed RDKit with bonds RMSD: 0.09769612853597351
CSV SCOPE maximum atom distance: 0.25838410701048214
Recomputed SCOPE maximum atom distance: 0.25838410701048214 at atom 1
CSV RDKit without bonds maximum atom distance: 0.25838410701048375
Recomputed RDKit without bonds maximum atom distance: 0.25838410701048375 for mobile/reference atom pair (1, 1)
CSV RDKit with bonds maximum atom distance: 0.25838410701048375
Recomputed RDKit with bonds maximum atom distance: 0.25838410701048375 for mobile/reference atom pair (1, 1)
XYZ folder: /Users/sergivela/Documents/SCOPE/Program/Scope/benchmarks/3-Molecule_Overlap/overlap_comparison_tests/Cel